# 02 — Polars: Expressions, Lazy Execution, Joins, Aggregation

**Overview:** reproduce the `01_pandas.ipynb` cleaning/groupby/merge pipeline using Polars'
expression API, then contrast eager vs lazy execution and inspect a query plan.

**Required libraries:** `polars`.

```
!pip install -q polars   # not preinstalled on Colab
```

Connects to: `01_pandas.ipynb` (same dataset, same operations). Leads to: `03_pyarrow.ipynb`
(the shared columnar format underneath both).

In [1]:
%pip install -q polars

Note: you may need to restart the kernel to use updated packages.


## 0. Environment & data

In [2]:
import sys
from pathlib import Path
import polars as pl

print("Python:", sys.version.split()[0])
print("polars:", pl.__version__)


DATA_PATH = Path("data/support_tickets.csv")

def generate_support_tickets_csv(path, n=300, seed=42):
    """Self-contained, deterministic synthetic dataset -- no external script or
    download needed, so this cell works the same in a fresh Colab runtime as it
    does locally."""
    import csv, random
    rng = random.Random(seed)
    categories = ["billing", "login", "performance", "bug", None]
    priorities = ["low", "medium", "high"]
    rows = []
    for i in range(n):
        rows.append({
            "ticket_id": i,
            "user_id": rng.randint(1, 50),
            "category": rng.choice(categories),
            "priority": rng.choice(priorities),
            "resolved": rng.choice([True, False]),
            "response_time_minutes": round(rng.uniform(5, 500), 1),
        })
    rows.extend(rows[:10])   # deterministic duplicates, used later to demo dedup handling
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

if not DATA_PATH.exists():
    generate_support_tickets_csv(DATA_PATH)

df = pl.read_csv(DATA_PATH)
print("shape:", df.shape)
df.head()

Python: 3.12.5
polars: 1.44.2
shape: (310, 6)


ticket_id,user_id,category,priority,resolved,response_time_minutes
i64,i64,str,str,bool,f64
0,41,"""billing""","""low""",false,126.2
1,9,"""billing""","""high""",true,297.3
2,3,"""billing""","""low""",true,120.2
3,39,"""billing""","""high""",true,359.4
4,45,null,"""medium""",true,227.4


## 1. The expression API

💡 **Key Idea:** `pl.col("x")` builds an *expression*, not a value — Polars can inspect, optimize,
and parallelize a whole chain of expressions before running anything.

In [3]:
high_priority = df.filter(pl.col("priority") == "high")
print("high priority tickets:", high_priority.height, "/", df.height)

slow_tickets = df.sort("response_time_minutes", descending=True).head(5)
print(slow_tickets.select(["ticket_id", "response_time_minutes"]))

assert (high_priority["priority"] == "high").all()


high priority tickets: 108 / 310
shape: (5, 2)
┌───────────┬───────────────────────┐
│ ticket_id ┆ response_time_minutes │
│ ---       ┆ ---                   │
│ i64       ┆ f64                   │
╞═══════════╪═══════════════════════╡
│ 129       ┆ 500.0                 │
│ 181       ┆ 499.8                 │
│ 246       ┆ 490.6                 │
│ 157       ┆ 490.4                 │
│ 79        ┆ 488.2                 │
└───────────┴───────────────────────┘


## 2. Missing values and duplicates

In [4]:
print("missing category BEFORE:", df["category"].is_null().sum())
print("duplicate rows BEFORE:", df.is_duplicated().sum())

rows_before = df.height
df_clean = df.unique(keep="first")
df_clean = df_clean.with_columns(pl.col("category").fill_null("unknown"))
rows_after = df_clean.height

print(f"rows before dedup: {rows_before}, after: {rows_after}")
print("missing category AFTER fillna:", df_clean["category"].is_null().sum())

assert rows_after < rows_before
assert df_clean["category"].is_null().sum() == 0


missing category BEFORE: 72
duplicate rows BEFORE: 20
rows before dedup: 310, after: 300
missing category AFTER fillna: 0


## 3. group_by + aggregation (expression style)

In [5]:
tickets_per_category = (
    df_clean.group_by("category")
    .agg(
        n_tickets=pl.col("ticket_id").count(),
        avg_response_time=pl.col("response_time_minutes").mean(),
        resolved_rate=pl.col("resolved").mean(),
    )
    .sort("n_tickets", descending=True)
)
print(tickets_per_category)
assert tickets_per_category["n_tickets"].sum() == df_clean.height


shape: (5, 4)
┌─────────────┬───────────┬───────────────────┬───────────────┐
│ category    ┆ n_tickets ┆ avg_response_time ┆ resolved_rate │
│ ---         ┆ ---       ┆ ---               ┆ ---           │
│ str         ┆ u32       ┆ f64               ┆ f64           │
╞═════════════╪═══════════╪═══════════════════╪═══════════════╡
│ unknown     ┆ 71        ┆ 255.056338        ┆ 0.478873      │
│ performance ┆ 61        ┆ 246.92459         ┆ 0.459016      │
│ login       ┆ 58        ┆ 281.262069        ┆ 0.568966      │
│ bug         ┆ 57        ┆ 275.931579        ┆ 0.421053      │
│ billing     ┆ 53        ┆ 271.615094        ┆ 0.584906      │
└─────────────┴───────────┴───────────────────┴───────────────┘


## 4. Joins

In [6]:
import random
rng = random.Random(0)
user_ids = sorted(df_clean["user_id"].unique().to_list())
region_lookup = pl.DataFrame({
    "user_id": user_ids,
    "region": [rng.choice(["us", "eu", "apac"]) for _ in user_ids],
})

joined = df_clean.join(region_lookup, on="user_id", how="left")
print("joined shape:", joined.shape)
assert joined.height == df_clean.height
assert "region" in joined.columns


joined shape: (300, 7)


## 5. Lazy execution

💡 **Key Idea:** `.lazy()` builds a query plan instead of running immediately; `.collect()`
triggers execution. `.explain()` shows the (possibly optimized) plan Polars will run —
e.g. filters get pushed down before a join/aggregation where possible.

In [7]:
lazy_query = (
    df_clean.lazy()
    .filter(pl.col("priority") != "low")
    .group_by("category")
    .agg(n_tickets=pl.col("ticket_id").count())
    .sort("n_tickets", descending=True)
)

print("Query plan:")
print(lazy_query.explain())

result = lazy_query.collect()     # nothing ran until this line
print("\nresult:\n", result)
assert result["n_tickets"].sum() <= df_clean.height


Query plan:
SORT BY [descending: [true]] [col("n_tickets")]
  AGGREGATE[maintain_order: false]
    [col("ticket_id").count().alias("n_tickets")] BY [col("category")]
    FROM
    simple π 2/2 ["category", "ticket_id"]
      FILTER col("priority") != "low"
      FROM
        DF ["ticket_id", "user_id", "category", "priority", ...]; PROJECT["category", "ticket_id", "priority"] 3/6 COLUMNS

result:
 shape: (5, 2)
┌─────────────┬───────────┐
│ category    ┆ n_tickets │
│ ---         ┆ ---       │
│ str         ┆ u32       │
╞═════════════╪═══════════╡
│ unknown     ┆ 52        │
│ login       ┆ 40        │
│ performance ┆ 39        │
│ bug         ┆ 38        │
│ billing     ┆ 35        │
└─────────────┴───────────┘


🔬 **Experiment:** eager vs lazy timing on this small dataset (the gap widens dramatically on
larger-than-RAM data, where lazy execution also enables streaming).

In [8]:
import time

start = time.perf_counter()
eager_result = (
    df_clean.filter(pl.col("priority") != "low")
    .group_by("category")
    .agg(n_tickets=pl.col("ticket_id").count())
)
eager_time = time.perf_counter() - start

start = time.perf_counter()
lazy_result = (
    df_clean.lazy()
    .filter(pl.col("priority") != "low")
    .group_by("category")
    .agg(n_tickets=pl.col("ticket_id").count())
    .collect()
)
lazy_time = time.perf_counter() - start

print(f"eager: {eager_time*1000:.3f} ms, lazy+collect: {lazy_time*1000:.3f} ms")
print("On tiny data the two are comparable; the lazy API's win is query optimization and "
      "streaming on data that doesn't fit in memory, not raw speed on 300 rows.")
assert eager_result.sort("category").equals(lazy_result.sort("category"))


eager: 0.716 ms, lazy+collect: 0.435 ms
On tiny data the two are comparable; the lazy API's win is query optimization and streaming on data that doesn't fit in memory, not raw speed on 300 rows.


## 6. Streaming (concept)

For data larger than RAM, Polars' lazy engine can execute a query plan in **streaming** batches
instead of materializing the full result at once (historically via `.collect(streaming=True)`;
the exact flag/engine name evolves across versions — always check your installed version's
docs). The mental model: the same `.lazy()...collect()` query plan above, but processed chunk by
chunk instead of all at once.

## 7. Connection to what's next

Both Pandas and Polars can share data with zero copies through Arrow's columnar memory format —
covered next in `03_pyarrow.ipynb`.

## 8. Interview questions
1. What does `.lazy()` buy you that eager execution doesn't?
2. What does `.explain()` show, and why is it useful before running a large query?
3. How does Polars' expression API (`pl.col(...)`) differ from Pandas' boolean-mask filtering style?
4. Why would streaming execution matter for a dataset larger than available RAM?
5. When would eager execution be preferable to lazy?

## 9. Exercises
1. Rewrite the Pandas pivot/melt from `01_pandas.ipynb` using Polars' `.pivot()`.
2. Build a lazy query that filters, joins with `region_lookup`, and aggregates — call `.explain()` before `.collect()`.
3. Compare the row count of a `group_by` result before and after adding an extra `.filter()` earlier in the lazy chain.

## 10. Summary

✅ Polars' expression API lets you write Pandas-equivalent logic that Polars can optimize via
lazy execution before running it — a core skill for data pipelines that outgrow a single
machine's RAM. Next: `03_pyarrow.ipynb` for the columnar format underneath both libraries.